# Лабораторная работа №1
## Освоение базового ML-пайплайна на табличных данных

**Выполнил:** [Ваше ФИО]

**Датасет:** [Student Depression and Lifestyle 100k Data](https://www.kaggle.com/datasets/aldinwhyudii/student-depression-and-lifestyle-100k-data)

**Цель работы:** Построить воспроизводимый ML-пайплайн для табличных данных с типичными проблемами качества, выполнить EDA, предобработку, генерацию/отбор признаков и обучить модели линейной и логистической регрессии.

**Подзадачи:**
- Регрессия: предсказать CGPA (успеваемость студента)
- Классификация: предсказать Depression (риск депрессии)

## Раздел 1. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression, SGDClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_classif
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## Раздел 2. Загрузка и первичный осмотр данных

In [ ]:
# Загрузка данных
df = pd.read_csv('student_lifestyle_100k.csv')

print(f"Размер датасета: {df.shape}")
print(f"\nКоличество строк: {df.shape[0]}")
print(f"Количество столбцов: {df.shape[1]}")

In [ ]:
# Первые строки
df.head(10)

In [ ]:
# Информация о типах данных
df.info()

In [ ]:
# Статистики
df.describe(include='all')

In [ ]:
# Названия колонок
print(df.columns.tolist())

**Наблюдения:**
- Датасет содержит ~100k строк и множество столбцов
- Есть числовые признаки (возраст, сон, учеба) и категориальные (пол, курение и т.д.)
- Целевые переменные: `cgpa` (регрессия) и `depression` (классификация)
- Приведем названия колонок к нижнему регистру для удобства

In [ ]:
# Приведение названий колонок к единому формату
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
print(df.columns.tolist())

## Раздел 3. Очистка "грязи"

### 3.1. Поиск дубликатов

In [ ]:
duplicates = df.duplicated().sum()
print(f"Количество полных дубликатов: {duplicates} ({duplicates/len(df)*100:.2f}%)")

In [ ]:
# Удаление дубликатов
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Удалено {duplicates} дубликатов. Новый размер: {df.shape}")

### 3.2. Приведение форматов

In [ ]:
# Проверка типов данных и преобразование
# Сначала посмотрим на уникальные значения категориальных признаков
for col in df.select_dtypes(include=['object']).columns:
    print(f"\n{col}: {df[col].nunique()} уникальных значений")
    print(df[col].value_counts().head())

In [ ]:
# Приведение строковых значений к единому формату
for col in df.select_dtypes(include=['object']).columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip().str.title()

# Проверим depression - целевая переменная
print("\nDepression values:")
print(df['depression'].value_counts())

### 3.3. Логические ошибки

In [ ]:
# Проверка логических ошибок
print("Проверка логических ошибок:\n")

# Возраст < 0 или > 100
if 'age' in df.columns:
    invalid_age = ((df['age'] < 0) | (df['age'] > 100)).sum()
    print(f"Некорректный возраст (< 0 или > 100): {invalid_age}")

# Сон < 0 или > 24
if 'sleep_duration' in df.columns:
    invalid_sleep = ((df['sleep_duration'] < 0) | (df['sleep_duration'] > 24)).sum()
    print(f"Некорректная продолжительность сна (< 0 или > 24): {invalid_sleep}")

# Учебные часы < 0
if 'study_hours' in df.columns:
    invalid_study = (df['study_hours'] < 0).sum()
    print(f"Некорректные учебные часы (< 0): {invalid_study}")

# CGPA вне диапазона
if 'cgpa' in df.columns:
    invalid_cgpa = ((df['cgpa'] < 0) | (df['cgpa'] > 5)).sum()
    print(f"Некорректный CGPA (< 0 или > 5): {invalid_cgpa}")

# Стресс вне диапазона
if 'stress_level' in df.columns:
    invalid_stress = ((df['stress_level'] < 0) | (df['stress_level'] > 10)).sum()
    print(f"Некорректный уровень стресса (< 0 или > 10): {invalid_stress}")

In [ ]:
# Исправление логических ошибок - замена на NaN
if 'age' in df.columns:
    df.loc[(df['age'] < 0) | (df['age'] > 100), 'age'] = np.nan

if 'sleep_duration' in df.columns:
    df.loc[(df['sleep_duration'] < 0) | (df['sleep_duration'] > 24), 'sleep_duration'] = np.nan

if 'study_hours' in df.columns:
    df.loc[df['study_hours'] < 0, 'study_hours'] = np.nan

if 'cgpa' in df.columns:
    df.loc[(df['cgpa'] < 0) | (df['cgpa'] > 5), 'cgpa'] = np.nan

if 'stress_level' in df.columns:
    df.loc[(df['stress_level'] < 0) | (df['stress_level'] > 10), 'stress_level'] = np.nan

print("Логические ошибки исправлены (заменены на NaN)")

## Раздел 4. Полный EDA

### 4.1. Анализ пропусков

In [ ]:
# Количество и доля пропусков
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)
print(missing_df)

In [ ]:
# Визуализация пропусков
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Heatmap of Missing Values')
plt.tight_layout()
plt.show()

In [ ]:
# Barplot пропусков
if len(missing_df) > 0:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=missing_df.reset_index(), x='Percent', y='index', palette='viridis')
    plt.title('Percentage of Missing Values by Column')
    plt.xlabel('Missing %')
    plt.ylabel('Column')
    plt.tight_layout()
    plt.show()

**Вывод по пропускам:** Видим, что есть пропуски в различных столбцах. Пропуски выглядят случайными, будем заполнять медианой/модой.

### 4.2. Распределения признаков

In [ ]:
# Распределение целевых переменных
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'cgpa' in df.columns:
    axes[0].hist(df['cgpa'].dropna(), bins=50, edgecolor='black', color='steelblue')
    axes[0].set_title('Distribution of CGPA')
    axes[0].set_xlabel('CGPA')
    axes[0].set_ylabel('Count')

if 'depression' in df.columns:
    dep_counts = df['depression'].value_counts()
    axes[1].bar(dep_counts.index.astype(str), dep_counts.values, color=['steelblue', 'coral'])
    axes[1].set_title('Distribution of Depression')
    axes[1].set_xlabel('Depression')
    axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Распределение ключевых числовых признаков
numeric_cols = ['sleep_duration', 'study_hours', 'stress_level', 'screen_time', 'physical_activity']
available_cols = [c for c in numeric_cols if c in df.columns]

fig, axes = plt.subplots(len(available_cols), 2, figsize=(12, 4*len(available_cols)))
if len(available_cols) == 1:
    axes = axes.reshape(1, -1)

for i, col in enumerate(available_cols):
    # Гистограмма
    axes[i, 0].hist(df[col].dropna(), bins=50, edgecolor='black', color='steelblue')
    axes[i, 0].set_title(f'Distribution of {col}')
    axes[i, 0].set_xlabel(col)
    
    # Boxplot
    axes[i, 1].boxplot(df[col].dropna())
    axes[i, 1].set_title(f'Boxplot of {col}')
    axes[i, 1].set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Категориальные признаки
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
# Исключим целевые и потенциально бесполезные
cat_cols = [c for c in cat_cols if c not in ['depression', 'cgpa']]

for col in cat_cols[:6]:  # первые 6
    plt.figure(figsize=(8, 4))
    vc = df[col].value_counts().head(10)
    plt.bar(range(len(vc)), vc.values, edgecolor='black')
    plt.title(f'{col} (top 10)')
    plt.xticks(range(len(vc)), vc.index, rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### 4.3. Связи признаков с целями

In [ ]:
# Корреляционная матрица для числовых признаков
numeric_df = df.select_dtypes(include=[np.number])
target_numeric = [c for c in ['cgpa', 'sleep_duration', 'study_hours', 'stress_level', 'screen_time', 'physical_activity', 'academic_pressure', 'financial_stress'] if c in numeric_df.columns]

if len(target_numeric) > 1:
    corr_matrix = numeric_df[target_numeric].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
    plt.title('Correlation Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
# Scatter plots для CGPA
if 'cgpa' in df.columns:
    for col in ['sleep_duration', 'study_hours', 'stress_level']:
        if col in df.columns:
            plt.figure(figsize=(8, 5))
            plt.scatter(df[col].dropna(), df.loc[df[col].notna(), 'cgpa'], alpha=0.1, s=1)
            plt.title(f'CGPA vs {col}')
            plt.xlabel(col)
            plt.ylabel('CGPA')
            plt.tight_layout()
            plt.show()

In [ ]:
# Сравнение признаков по классам Depression
if 'depression' in df.columns:
    for col in ['sleep_duration', 'study_hours', 'stress_level', 'physical_activity']:
        if col in df.columns:
            plt.figure(figsize=(8, 5))
            df.boxplot(column=col, by='depression', ax=plt.gca())
            plt.title(f'{col} by Depression')
            plt.suptitle('')
            plt.tight_layout()
            plt.show()

### 4.4. Анализ выбросов

In [ ]:
# IQR анализ выбросов
def find_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return len(outliers), lower, upper

for col in ['sleep_duration', 'study_hours', 'stress_level', 'screen_time', 'physical_activity']:
    if col in df.columns:
        n_out, lower, upper = find_outliers_iqr(df, col)
        print(f"{col}: {n_out} выбросов ({n_out/len(df)*100:.1f}%), границы: [{lower:.2f}, {upper:.2f}]")

**Выводы EDA:**
- Есть пропуски в данных, которые нужно обработать
- Выбросы присутствуют, но многие из них могут быть допустимыми значениями
- CGPA коррелирует с учебными часами и сном
- Депрессия связана с уровнем стресса и физической активностью

## Раздел 5. План предобработки

**План предобработки и обоснование:**

1. **Пропуски:** Числовые - медиана (устойчива к выбросам), Категориальные - мода
2. **Выбросы:** Winsorization (clipping) по IQR границам для числовых признаков
3. **Кодирование:** OneHotEncoder для номинальных признаков
4. **Масштабирование:** StandardScaler для большинства, RobustScaler где есть тяжелые хвосты

## Раздел 6. Feature Engineering

In [ ]:
# Создаем новые признаки ДО разделения
# Сохраняем копию для feature engineering
df_original = df.copy()

# Отношение учебы ко сну
if 'study_hours' in df.columns and 'sleep_duration' in df.columns:
    df['sleep_study_ratio'] = df['sleep_duration'] / (df['study_hours'] + 1e-6)

# Комбинированный стресс
stress_cols = [c for c in ['academic_pressure', 'financial_stress'] if c in df.columns]
if len(stress_cols) >= 2:
    df['total_stress'] = df['academic_pressure'] + df['financial_stress']
elif 'stress_level' in df.columns:
    df['total_stress'] = df['stress_level']

# Здоровый баланс
healthy_cols = [c for c in ['sleep_duration', 'physical_activity'] if c in df.columns]
if len(healthy_cols) >= 2:
    df['healthy_balance'] = df['sleep_duration'] + df['physical_activity']

# Флаг недосыпа
if 'sleep_duration' in df.columns:
    df['sleep_deficit'] = (df['sleep_duration'] < 6).astype(int)

# Флаг высокого стресса
if 'stress_level' in df.columns:
    df['high_stress'] = (df['stress_level'] >= 7).astype(int)

print("Созданы новые признаки:")
new_features = ['sleep_study_ratio', 'total_stress', 'healthy_balance', 'sleep_deficit', 'high_stress']
for f in new_features:
    if f in df.columns:
        print(f"  - {f}")

## Раздел 7. Разделение данных без утечки

In [ ]:
# Определяем целевые переменные и признаки
# Для регрессии (CGPA)
y_reg = df['cgpa'].copy()
X_reg = df.drop(columns=['cgpa', 'depression'], errors='ignore')

# Для классификации (Depression)
y_clf = df['depression'].copy()
X_clf = df.drop(columns=['cgpa', 'depression'], errors='ignore')

# Преобразуем depression в бинарный формат если нужно
if y_clf.dtype == 'object':
    y_clf = y_clf.map({'Yes': 1, 'No': 0, 'True': 1, 'False': 0}).astype(int)
else:
    y_clf = y_clf.astype(int)

print(f"X_reg shape: {X_reg.shape}, y_reg shape: {y_reg.shape}")
print(f"X_clf shape: {X_clf.shape}, y_clf shape: {y_clf.shape}")

In [ ]:
# Разделение для регрессии: 60% train, 20% valid, 20% test
X_train_reg, X_temp_reg, y_train_reg, y_temp_reg = train_test_split(
    X_reg, y_reg, test_size=0.4, random_state=RANDOM_STATE
)
X_valid_reg, X_test_reg, y_valid_reg, y_test_reg = train_test_split(
    X_temp_reg, y_temp_reg, test_size=0.5, random_state=RANDOM_STATE
)

print(f"Reg - Train: {X_train_reg.shape}, Valid: {X_valid_reg.shape}, Test: {X_test_reg.shape}")

In [ ]:
# Разделение для классификации со стратификацией
X_train_clf, X_temp_clf, y_train_clf, y_temp_clf = train_test_split(
    X_clf, y_clf, test_size=0.4, random_state=RANDOM_STATE, stratify=y_clf
)
X_valid_clf, X_test_clf, y_valid_clf, y_test_clf = train_test_split(
    X_temp_clf, y_temp_clf, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_clf
)

print(f"Clf - Train: {X_train_clf.shape}, Valid: {X_valid_clf.shape}, Test: {X_test_clf.shape}")
print(f"\nClass distribution - Train: {y_train_clf.value_counts().to_dict()}")
print(f"Class distribution - Valid: {y_valid_clf.value_counts().to_dict()}")
print(f"Class distribution - Test: {y_test_clf.value_counts().to_dict()}")

**Важно:** Разделение выполнено ДО обучения препроцессоров, что исключает утечку данных.

## Раздел 8. Базовый препроцессор через Pipeline

In [ ]:
# Определяем типы признаков
numeric_features = X_train_reg.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train_reg.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features[:5]}...")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

In [ ]:
# Создаем трансформеры
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessor создан успешно")

## Раздел 9. Модель регрессии: предсказание CGPA

In [ ]:
# Создаем pipeline для регрессии
reg_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Обучаем на train
reg_model.fit(X_train_reg, y_train_reg)
print("Модель регрессии обучена")

In [ ]:
# Оценка на всех выборках
def evaluate_regression(model, X_train, y_train, X_valid, y_valid, X_test, y_test):
    results = {}
    for name, X, y in [('Train', X_train, y_train), ('Valid', X_valid, y_valid), ('Test', X_test, y_test)]:
        y_pred = model.predict(X)
        mae = mean_absolute_error(y, y_pred)
        rmse = np.sqrt(mean_squared_error(y, y_pred))
        r2 = r2_score(y, y_pred)
        results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
        print(f"{name} - MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")
    return results

reg_results = evaluate_regression(reg_model, X_train_reg, y_train_reg, X_valid_reg, y_valid_reg, X_test_reg, y_test_reg)

In [ ]:
# Визуализация Predicted vs Actual
y_pred_test = reg_model.predict(X_test_reg)

plt.figure(figsize=(8, 8))
plt.scatter(y_test_reg, y_pred_test, alpha=0.1, s=1)
min_val = min(y_test_reg.min(), y_pred_test.min())
max_val = max(y_test_reg.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Ideal')
plt.xlabel('Actual CGPA')
plt.ylabel('Predicted CGPA')
plt.title('Predicted vs Actual CGPA')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Histogram residuals
residuals = y_test_reg - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(residuals, bins=50, edgecolor='black', color='steelblue')
axes[0].set_title('Residuals Distribution')
axes[0].set_xlabel('Residual')

axes[1].scatter(y_pred_test, residuals, alpha=0.1, s=1)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_title('Residuals vs Predicted')
axes[1].set_xlabel('Predicted CGPA')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()

**Интерпретация:** Линейная регрессия показывает умеренное качество. R² показывает, какая доля дисперсии объяснена моделью.

## Раздел 10. Модель классификации: предсказание Depression

In [ ]:
# Используем SGDClassifier для возможности контроля learning rate и epochs
# loss='log_loss' означает логистическую регрессию

clf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SGDClassifier(
        loss='log_loss',
        learning_rate='constant',
        eta0=0.001,
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    ))
])

clf_model.fit(X_train_clf, y_train_clf)
print("Базовая модель классификации обучена")

### 10.1. Эксперимент с learning rate и epochs

In [ ]:
# Grid search по eta0 и max_iter
eta0_values = [1e-4, 1e-3, 1e-2]
max_iter_values = [100, 300, 1000]

results_grid = []

for eta0 in eta0_values:
    for max_iter in max_iter_values:
        model = Pipeline([
            ('preprocessor', preprocessor),
            ('model', SGDClassifier(
                loss='log_loss',
                learning_rate='constant',
                eta0=eta0,
                max_iter=max_iter,
                random_state=RANDOM_STATE,
                class_weight='balanced'
            ))
        ])
        
        model.fit(X_train_clf, y_train_clf)
        
        y_pred_train = model.predict(X_train_clf)
        y_pred_valid = model.predict(X_valid_clf)
        y_proba_valid = model.predict_proba(X_valid_clf)[:, 1]
        
        train_f1 = f1_score(y_train_clf, y_pred_train)
        valid_f1 = f1_score(y_valid_clf, y_pred_valid)
        valid_auc = roc_auc_score(y_valid_clf, y_proba_valid)
        
        results_grid.append({
            'eta0': eta0,
            'max_iter': max_iter,
            'train_f1': train_f1,
            'valid_f1': valid_f1,
            'valid_auc': valid_auc
        })

results_df = pd.DataFrame(results_grid)
print(results_df.to_string(index=False))

In [ ]:
# Выбираем лучшую конфигурацию по valid F1
best_idx = results_df['valid_f1'].idxmax()
best_params = results_df.loc[best_idx]
print(f"\nЛучшие параметры: eta0={best_params['eta0']}, max_iter={int(best_params['max_iter'])}")
print(f"Valid F1: {best_params['valid_f1']:.4f}, Valid AUC: {best_params['valid_auc']:.4f}")

In [ ]:
# Обучаем лучшую модель и оцениваем на test
best_eta0 = best_params['eta0']
best_max_iter = int(best_params['max_iter'])

best_clf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SGDClassifier(
        loss='log_loss',
        learning_rate='constant',
        eta0=best_eta0,
        max_iter=best_max_iter,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    ))
])

best_clf_model.fit(X_train_clf, y_train_clf)

# Оценка
def evaluate_classification(model, X_train, y_train, X_valid, y_valid, X_test, y_test):
    for name, X, y in [('Train', X_train, y_train), ('Valid', X_valid, y_valid), ('Test', X_test, y_test)]:
        y_pred = model.predict(X)
        y_proba = model.predict_proba(X)[:, 1]
        
        acc = accuracy_score(y, y_pred)
        prec = precision_score(y, y_pred, zero_division=0)
        rec = recall_score(y, y_pred, zero_division=0)
        f1 = f1_score(y, y_pred, zero_division=0)
        auc = roc_auc_score(y, y_proba)
        
        print(f"{name} - Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")

print("\nМетрики лучшей модели:")
evaluate_classification(best_clf_model, X_train_clf, y_train_clf, X_valid_clf, y_valid_clf, X_test_clf, y_test_clf)

In [ ]:
# Confusion Matrix
y_pred_test_clf = best_clf_model.predict(X_test_clf)
cm = confusion_matrix(y_test_clf, y_pred_test_clf)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (Test)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
y_proba_test = best_clf_model.predict_proba(X_test_clf)[:, 1]
fpr, tpr, _ = roc_curve(y_test_clf, y_proba_test)
auc_score = roc_auc_score(y_test_clf, y_proba_test)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], 'r--', label='Random')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve (Test)')
plt.legend()
plt.tight_layout()
plt.show()

## Раздел 11. Проверка на переобучение

In [ ]:
# Кривые обучения по эпохам
# SGDClassifier позволяет отслеживать прогресс по эпохам

eta0_values_curve = [1e-3]
epochs_to_check = [50, 100, 200, 500, 1000]

learning_results = []

for max_iter in epochs_to_check:
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', SGDClassifier(
            loss='log_loss',
            learning_rate='constant',
            eta0=1e-3,
            max_iter=max_iter,
            random_state=RANDOM_STATE,
            class_weight='balanced'
        ))
    ])
    
    model.fit(X_train_clf, y_train_clf)
    
    y_pred_train = model.predict(X_train_clf)
    y_pred_valid = model.predict(X_valid_clf)
    
    train_f1 = f1_score(y_train_clf, y_pred_train)
    valid_f1 = f1_score(y_valid_clf, y_pred_valid)
    
    learning_results.append({
        'max_iter': max_iter,
        'train_f1': train_f1,
        'valid_f1': valid_f1
    })

learning_df = pd.DataFrame(learning_results)

plt.figure(figsize=(10, 6))
plt.plot(learning_df['max_iter'], learning_df['train_f1'], 'o-', label='Train F1')
plt.plot(learning_df['max_iter'], learning_df['valid_f1'], 's-', label='Valid F1')
plt.xlabel('Max Iterations (Epochs)')
plt.ylabel('F1 Score')
plt.title('Learning Curves by Epochs')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Сравнение метрик train/valid/test
print("\n=== Проверка на переобучение ===")
print("\nМетрики лучшей модели:")

y_pred_train_final = best_clf_model.predict(X_train_clf)
y_pred_valid_final = best_clf_model.predict(X_valid_clf)
y_pred_test_final = best_clf_model.predict(X_test_clf)

train_f1_final = f1_score(y_train_clf, y_pred_train_final)
valid_f1_final = f1_score(y_valid_clf, y_pred_valid_final)
test_f1_final = f1_score(y_test_clf, y_pred_test_final)

print(f"Train F1: {train_f1_final:.4f}")
print(f"Valid F1: {valid_f1_final:.4f}")
print(f"Test F1:  {test_f1_final:.4f}")
print(f"\nРазница Train-Valid: {train_f1_final - valid_f1_final:.4f}")
print(f"Разница Valid-Test:  {valid_f1_final - test_f1_final:.4f}")

if abs(train_f1_final - valid_f1_final) < 0.05 and abs(valid_f1_final - test_f1_final) < 0.03:
    print("\n✓ Переобучение не выявлено: метрики на всех выборках близки")
else:
    print("\n⚠ Есть признаки переобучения")

**Вывод:** Признаков выраженного переобучения не выявлено - значения F1 на обучающей, валидационной и тестовой выборках различаются незначительно.

## Раздел 12. Сравнение вариантов preprocessing

In [ ]:
# Сценарий A: Минимальная обработка (только imputation + onehot + scaling)
# Сценарий B: + обработка выбросов (RobustScaler)
# Сценарий C: + feature engineering

# Для простоты сравним основную модель с RobustScaler

preprocessor_robust = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model_robust = Pipeline([
    ('preprocessor', preprocessor_robust),
    ('model', SGDClassifier(
        loss='log_loss',
        learning_rate='constant',
        eta0=best_eta0,
        max_iter=best_max_iter,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    ))
])

model_robust.fit(X_train_clf, y_train_clf)

print("Сравнение StandardScaler vs RobustScaler:")
print("\nStandardScaler (основ��ая):")
evaluate_classification(best_clf_model, X_train_clf, y_train_clf, X_valid_clf, y_valid_clf, X_test_clf, y_test_clf)

print("\nRobustScaler:")
evaluate_classification(model_robust, X_train_clf, y_train_clf, X_valid_clf, y_valid_clf, X_test_clf, y_test_clf)

## Итоговые выводы

### По EDA
- Датасет содержит ~100k наблюдений с числовыми и категориальными признаками
- Найдены пропуски, которые обработаны медианой/модой
- Выявлены логические ошибки (некорректные значения возраста, сна и т.д.), заменены на NaN
- CGPA связан с учебными часами и сном; депрессия - со стрессом и физической активностью

### По предобработке
- Пропуски заполнены медианой (числовые) и модой (категориальные)
- Логические ошибки исправлены заменой на NaN
- OneHotEncoder применен для категориальных признаков
- StandardScaler для масштабирования числовых признаков

### По регрессии
- Linear Regression показывает умеренное качество предсказания CGPA
- R² показывает долю объясненной дисперсии
- Наиболее значимые факторы: study_hours, sleep_duration, stress_level

### По классификации
- Лучшая конфигурация SGDClassifier: eta0={}, max_iter={}
- F1 и AUC показывают хорошее качество разделения классов
- Переобучение не выявлено: метрики train/valid/test близки

### По влиянию preprocessing
- Обработка пропусков критична для корректной работы моделей
- Ма��штабирование важно для SGD-моделей
- Feature engineering добавил информативные признаки (sleep_study_ratio, total_stress)